# Second model (goal=0.127992 (top 1000))
im impementing pipelines and XGboost on this model

## Equal to model 1 (load train set)

In [2]:
# Librerías para manipulación de datos y visualización
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from random import randint
from sklearn.metrics import mean_squared_log_error

# Configuración para que se muestren bien los gráficos
%matplotlib inline
print("Librerías importadas correctamente.")




Librerías importadas correctamente.


In [3]:
 # Cargar los datos
df = pd.read_csv('../data/train.csv', index_col='Id')

# Mostrar las primeras 5 filas para confirmar la carga
print("Primeras 5 filas del dataset:")
df

Primeras 5 filas del dataset:


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500


In [4]:
df_to_pred = pd.read_csv('../data/test.csv', index_col='Id')
df_to_pred_num=df_to_pred.select_dtypes(include=['float64', 'int64'])
df_to_pred_cat=df_to_pred.select_dtypes(exclude=['float64', 'int64'])
df_to_pred

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml


## First thing i will do is a super linear model including all varibales

In [5]:
#hago un dataframe numerico
dfNum=df.select_dtypes(include=['float64', 'int64'])
#extraigo las columnas que no son numéricas
dfCat = df.select_dtypes(exclude=['float64', 'int64'])
Categoric=dfCat.columns.tolist()


In [6]:

def modelmaker(df,y,rnd="na"):
    formodeldf=df.copy()
    
    if rnd=="na":
        rnd=randint(1,500)

    x_train, x_test, y_train, y_test = train_test_split(formodeldf,y, test_size=0.2, random_state=rnd)


    reg=LinearRegression()
    reg.fit(x_train,y_train)
    y_hat=reg.predict(x_test)
    
    # scores=pd.DataFrame(data={'Medida':["MAE","MSE","RMSE","R2"],'Valor':[mae,mse,rmse,r2]})
    # scores.style.format({'Valor': '{:.2f}'})

    #get rid of 0's for RMSLE
    y_pred_no0 = np.maximum(y_hat, 0)
    y_test_no0 = np.maximum(y_test, 0)

    rmsle = np.sqrt(mean_squared_log_error(y_test_no0, y_pred_no0))
    #mod is for returning the model intead of metrics


    return rmsle, reg


In [7]:
father_df=df.copy()
nonadf=father_df.copy()
nonadf[dfNum.columns] = nonadf[dfNum.columns].fillna(nonadf[dfNum.columns].median())
nonadf

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500


In [8]:
dummies = pd.get_dummies(nonadf, drop_first=True,)
dummies

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
Id,,,,,,,,,,,,,,,,,,,,,
1,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,False,False,False,False,True,False,False,False,True,False
2,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,False,False,False,False,True,False,False,False,True,False
3,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,False,False,False,False,True,False,False,False,True,False
4,70,60.0,9550,7,5,1915,1970,0.0,216,0,...,False,False,False,False,True,False,False,False,False,False
5,60,84.0,14260,8,5,2000,2000,350.0,655,0,...,False,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,62.0,7917,6,5,1999,2000,0.0,0,0,...,False,False,False,False,True,False,False,False,True,False
1457,20,85.0,13175,6,6,1978,1988,119.0,790,163,...,False,False,False,False,True,False,False,False,True,False
1458,70,66.0,9042,7,9,1941,2006,0.0,275,0,...,False,False,False,False,True,False,False,False,True,False


In [9]:
forcorrdf_to_pred=dfNum.copy()

zeromeanslack=['GarageArea', 'TotalBsmtSF', 'MasVnrArea', 'BsmtFinSF1', 'WoodDeckSF','2ndFlrSF', 'OpenPorchSF', 'BsmtUnfSF', 'EnclosedPorch','ScreenPorch','PoolArea','3SsnPorch','LowQualFinSF','MiscVal','BsmtFinSF2']
# I selected this features bc 0's in them mean lacking of the feature, 

# #new corr
# corr_matrix= forcorrdfnum.corr()['SalePrice'].drop('SalePrice')
# new_corr= {}

# #calc corr with saleprice for each feature independently (THIS ACTUUALLY IS NOT THAT USEFULL XD)
# for column in zeromeanslack:
#     new_corr[column]=forcorrdfnum[forcorrdfnum[column]!=0].corr()['SalePrice'][column]

# #
# dic_ordenado = dict(sorted(new_corr.items(), key=lambda x: x[1], reverse=True))

# #substituing old with new correlations
# for i in corr_matrix.index:
#     if i in zeromeanslack:
#         corr_matrix[i] = dic_ordenado[i]

# #ordering
# corr_matrix_2=corr_matrix.sort_values(ascending=False)
# print(corr_matrix_2)
# #THIS IS A MORE PRECIZE CORRELATION MATRIX

In [10]:
def addhascolumns(df,columnames):
    dfforfunc=df.copy()
    for column in columnames:
        if column in dfforfunc.columns:
            #for each column to clean if is in the df, create a has feature with 0 if 0 and 1 if >0
            dfforfunc[f'has {column}']=(dfforfunc[column]>0).astype(int)
        else:
            print(f"{column} wasn't found")
    return dfforfunc

In [11]:
dfformegamodel=addhascolumns(dummies,zeromeanslack)
dfformegamodel

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,has 2ndFlrSF,has OpenPorchSF,has BsmtUnfSF,has EnclosedPorch,has ScreenPorch,has PoolArea,has 3SsnPorch,has LowQualFinSF,has MiscVal,has BsmtFinSF2
Id,,,,,,,,,,,,,,,,,,,,,
1,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,1,1,1,0,0,0,0,0,0,0
2,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,0,0,1,0,0,0,0,0,0,0
3,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,1,1,1,0,0,0,0,0,0,0
4,70,60.0,9550,7,5,1915,1970,0.0,216,0,...,1,1,1,1,0,0,0,0,0,0
5,60,84.0,14260,8,5,2000,2000,350.0,655,0,...,1,1,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,62.0,7917,6,5,1999,2000,0.0,0,0,...,1,1,1,0,0,0,0,0,0,0
1457,20,85.0,13175,6,6,1978,1988,119.0,790,163,...,0,0,1,0,0,0,0,0,0,1
1458,70,66.0,9042,7,9,1941,2006,0.0,275,0,...,1,1,1,0,0,0,0,0,1,0


In [12]:
yformegamodel=dfformegamodel['SalePrice']
dfformegamodel=dfformegamodel.drop('SalePrice', axis=1)


In [13]:
supermodel=modelmaker(dfformegamodel,yformegamodel,42)[1]

In [14]:
df_to_pred['SalePrice']=0
df_to_pred[dfNum.columns] = df_to_pred[dfNum.columns].fillna(df_to_pred[dfNum.columns].median())


x_predformegamodel=pd.get_dummies(df_to_pred, drop_first=True,)
x_predformegamodel.drop('SalePrice', axis=1)
x_predformegamodel=addhascolumns(x_predformegamodel,zeromeanslack)

x_predformegamodel

unseenidkwhy=list(set(dfformegamodel.columns) - set(x_predformegamodel.columns))

x_predformegamodel[unseenidkwhy]=0

x_predformegamodel = x_predformegamodel[dfformegamodel.columns]



In [15]:
def createsubmissionfie(name,model,x_to_pred):


    import joblib
    joblib.dump(model, f'Model_{name}.joblib')


    pred_test=model.predict(x_to_pred)
    submission = pd.DataFrame({
        'Id': x_to_pred.index,
        'SalePrice': pred_test
    })

    # Guardar el archivo
    submission.to_csv(f'Submission_{name}.csv', index=False)
    print("Archivo de submission creado exitosamente!")

In [16]:
createsubmissionfie("1_9",supermodel,x_predformegamodel)

Archivo de submission creado exitosamente!


so this was quite rustic so, now im going to select some specific variables fo de dummies, adn use my previus best model 

firts im getting a correlation matrix with saleprice

In [17]:

#corrmatrix
corr_matrix= dfNum.corr()['SalePrice'].drop('SalePrice')

#calc corr with saleprice for each feature independently (THIS ACTUUALLY IS NOT THAT USEFULL XD)
for column in zeromeanslack:
    corr_matrix[column]=dfNum[dfNum[column]!=0].corr()['SalePrice'][column]
corr_matrix.sort_values(ascending=False)

OverallQual      0.790982
GrLivArea        0.708624
2ndFlrSF         0.673305
GarageCars       0.640409
TotalBsmtSF      0.609681
GarageArea       0.608405
1stFlrSF         0.605852
FullBath         0.560664
TotRmsAbvGrd     0.533723
YearBuilt        0.522897
YearRemodAdd     0.507101
GarageYrBlt      0.486362
BsmtFinSF1       0.471690
Fireplaces       0.466929
MasVnrArea       0.434090
LotFrontage      0.351799
LowQualFinSF     0.300075
HalfBath         0.284108
LotArea          0.263843
ScreenPorch      0.255430
EnclosedPorch    0.241279
BsmtFullBath     0.227122
BsmtFinSF2       0.198956
WoodDeckSF       0.193706
BsmtUnfSF        0.169261
BedroomAbvGr     0.168213
MiscVal          0.088963
OpenPorchSF      0.086453
3SsnPorch        0.063932
MoSold           0.046432
PoolArea        -0.014092
BsmtHalfBath    -0.016844
YrSold          -0.028923
OverallCond     -0.077856
MSSubClass      -0.084284
KitchenAbvGr    -0.135907
Name: SalePrice, dtype: float64

From categorical vars i selected some that semeed to provide more variance

In [18]:
selectctfd=dfCat.copy()

selectctfd['SalePrice']=dfNum['SalePrice']

selectctfd = selectctfd[selectctfd['SalePrice'] < selectctfd['SalePrice'].quantile(0.95)]


In [19]:
# cat_per_column=pd.DataFrame()
# for colum in selectctfd.drop('SalePrice',axis=1).columns:
#     i=0
#     sscolumn=pd.Series()
#     for cat in selectctfd[colum].unique():
#         sscolumn[i]=selectctfd[selectctfd[colum]==cat]['SalePrice'].median()
#         i+=1
#     cat_per_column = pd.concat([cat_per_column, sscolumn.rename(colum)], axis=1)
cat_per_column=pd.DataFrame()
ns_per_column=pd.DataFrame()
for colum in selectctfd.drop('SalePrice',axis=1).columns:
    i=0
    sscolumn=pd.Series()
    
    nanns=selectctfd[colum].isna().sum()
    if nanns:
        restserie=np.append(selectctfd[colum].value_counts().values,nanns)
        restserie=pd.Series(restserie)
    else:
        restserie=selectctfd[colum].value_counts().values
        restserie=pd.Series(restserie)

    for cat in selectctfd[colum].unique():
        sscolumn[i]=selectctfd[selectctfd[colum]==cat]['SalePrice'].median()
        i+=1
    cat_per_column = pd.concat([cat_per_column, sscolumn.rename(colum)], axis=1)
    ns_per_column = pd.concat([ns_per_column, restserie.rename(colum)], axis=1)

In [20]:
cat_per_column.head()

,MSZoning,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,...,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,PoolQC,Fence,MiscFeature,SaleType,SaleCondition
0,169945.0,159000.0,NaN,144500.0,158000.0,159000.0,155000.0,158000.0,195950.0,161750.0,...,180000.0,187500.0,165000.0,165000.0,164000.0,NaN,NaN,NaN,155000.0,157500.0
1,120250.0,114250.0,119500.0,184000.0,139400.0,137500.0,164000.0,170000.0,206000.0,140000.0,...,129500.0,135000.0,115000.0,114504.0,111000.0,235000.0,137000.0,144000.0,220750.0,130000.0
2,74700.0,NaN,172500.0,195000.0,188700.0,NaN,158500.0,182500.0,191000.0,198750.0,...,108000.0,194201.0,209115.0,NaN,132250.0,215500.0,138500.0,170750.0,139000.0,220000.0
3,197000.0,NaN,NaN,192140.0,188750.0,NaN,191000.0,NaN,270395.0,119200.0,...,214950.0,NaN,NaN,148000.0,NaN,171000.0,166250.0,94000.0,140000.0,104000.0
4,136500.0,NaN,NaN,NaN,NaN,NaN,195450.0,NaN,153500.0,142500.0,...,NaN,NaN,96500.0,108000.0,NaN,NaN,130000.0,250000.0,120000.0,142953.0


In [21]:
minusmean=cat_per_column-cat_per_column.mean()
catordered=minusmean.pow(2).sum().sort_values(ascending=False)
catordered

Neighborhood          61013539204.0
Exterior1st           42028924000.0
Exterior2nd      37066070100.234375
Condition2          36921826021.875
ExterQual             24176322500.0
SaleType         20304573888.888885
RoofMatl              18225718750.0
BsmtQual             15661962968.75
KitchenQual          14101157018.75
MiscFeature           12739671875.0
FireplaceQu           11158800000.0
BsmtCond               9063687500.0
MSZoning               8811590320.0
GarageQual             8156643580.0
Condition1             7897037200.0
SaleCondition     7607914840.833333
GarageType             7155968750.0
ExterCond              6297300000.0
Heating           6027020520.833334
Electrical             5809550000.0
HeatingQC              5591792000.0
HouseStyle           5500589446.875
Foundation        5274642083.333333
RoofStyle         4408808333.333334
MasVnrType             3798000000.0
Functional        3226383571.428572
BsmtFinType1      2852583333.333333
BsmtFinType2           24430

In [22]:
ns_per_column

,MSZoning,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,...,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,PoolQC,Fence,MiscFeature,SaleType,SaleCondition
0,1084.0,1381.0,50.0,893.0,1249.0,1386.0,1001.0,1316.0,224,1191.0,...,815.0,602.0,1239.0,1253.0,1267.0,3.0,155.0,49.0,1228.0,1160.0
1,216.0,6.0,41.0,450.0,63.0,1.0,252.0,59.0,148,81.0,...,386.0,405.0,48.0,35.0,90.0,2.0,58.0,2.0,90.0,99.0
2,61.0,NaN,1296.0,35.0,41.0,NaN,85.0,12.0,112,47.0,...,72.0,299.0,14.0,9.0,30.0,1.0,53.0,2.0,43.0,93.0
3,16.0,NaN,NaN,9.0,34.0,NaN,45.0,NaN,100,25.0,...,18.0,81.0,3.0,7.0,NaN,1381.0,11.0,1.0,9.0,20.0
4,10.0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,81,18.0,...,9.0,NaN,2.0,2.0,NaN,NaN,1110.0,1333.0,5.0,11.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78,11.0,...,6.0,NaN,81.0,81.0,NaN,NaN,NaN,NaN,4.0,4.0
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74,7.0,...,81.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
distancemetric=1/(ns_per_column.pow(2).sum())
distancemetric.sort_values(ascending=False)

Neighborhood     7.264116e-06
Exterior2nd      2.766382e-06
Exterior1st      2.654358e-06
BsmtFinType1     2.510450e-06
GarageFinish     1.606707e-06
FireplaceQu      1.480628e-06
HouseStyle       1.479511e-06
HeatingQC        1.441032e-06
Foundation       1.326811e-06
BsmtQual         1.277705e-06
GarageType       1.211524e-06
KitchenQual      1.173126e-06
MasVnrType       1.091063e-06
BsmtExposure     1.061268e-06
LotShape         9.987466e-07
ExterQual        9.843634e-07
LotConfig        9.304308e-07
MSZoning         8.158011e-07
Fence            7.921300e-07
RoofStyle        7.740064e-07
BldgType         7.460821e-07
SaleCondition    7.328225e-07
BsmtFinType2     7.069791e-07
Condition1       7.000845e-07
ExterCond        6.691331e-07
SaleType         6.587281e-07
GarageQual       6.475866e-07
BsmtCond         6.452933e-07
LandContour      6.382407e-07
GarageCond       6.337421e-07
Electrical       6.251184e-07
PavedDrive       6.194678e-07
Functional       6.018734e-07
CentralAir

In [24]:
ord_bsd_on_median_ns=(catordered*distancemetric).sort_values(ascending=False)
ord_bsd_on_median_ns

Neighborhood     443209.425946
Exterior1st       111559.79073
Exterior2nd      102538.902522
ExterQual         23798.286716
BsmtQual          20011.375372
Condition2        19585.336561
KitchenQual       16542.440806
FireplaceQu       16522.033969
SaleType          13375.193675
RoofMatl           9795.324093
GarageType           8669.6245
HouseStyle         8138.182549
HeatingQC          8057.952553
MSZoning           7188.504971
BsmtFinType1       7161.267108
MiscFeature         7159.93876
Foundation         6998.451722
BsmtCond           5848.736484
SaleCondition      5575.250857
Condition1         5528.593341
GarageQual         5282.133137
ExterCond          4213.731815
MasVnrType         4143.858895
Electrical         3631.656495
RoofStyle          3412.445816
GarageFinish       3377.250957
Heating            3277.166498
Functional         1941.874421
BsmtFinType2       1727.206243
LotShape            1641.81672
GarageCond         1451.941635
LotConfig          1342.641363
CentralA

In [25]:
numerical_2=dfNum.copy()
y_categorical_selction=numerical_2['SalePrice']
numerical_2=numerical_2.drop(['SalePrice','1stFlrSF', 'YearBuilt', 'GarageYrBlt', 'LotFrontage', 'HalfBath', 'ScreenPorch', 'EnclosedPorch', 'BsmtFullBath', 'OpenPorchSF', '3SsnPorch', 'MoSold', 'PoolArea', 'YrSold', 'KitchenAbvGr'], axis=1)
numerical_2=addhascolumns(numerical_2,zeromeanslack)
numerical_2=numerical_2.fillna(numerical_2.median())
numerical_2

OpenPorchSF wasn't found
EnclosedPorch wasn't found
ScreenPorch wasn't found
PoolArea wasn't found
3SsnPorch wasn't found


,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,has GarageArea,has TotalBsmtSF,has MasVnrArea,has BsmtFinSF1,has WoodDeckSF,has 2ndFlrSF,has BsmtUnfSF,has LowQualFinSF,has MiscVal,has BsmtFinSF2
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,196.0,706,0,150,856,...,1,1,1,1,0,1,1,0,0,0
2,20,9600,6,8,1976,0.0,978,0,284,1262,...,1,1,0,1,1,0,1,0,0,0
3,60,11250,7,5,2002,162.0,486,0,434,920,...,1,1,1,1,0,1,1,0,0,0
4,70,9550,7,5,1970,0.0,216,0,540,756,...,1,1,0,1,0,1,1,0,0,0
5,60,14260,8,5,2000,350.0,655,0,490,1145,...,1,1,1,1,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,6,5,2000,0.0,0,0,953,953,...,1,1,0,0,0,1,1,0,0,0
1457,20,13175,6,6,1988,119.0,790,163,589,1542,...,1,1,1,1,1,0,1,0,0,1
1458,70,9042,7,9,2006,0.0,275,0,877,1152,...,1,1,0,1,0,1,1,0,1,0


In [26]:
scorenum=[]
for i in range(1,1000):
    scorenum.append(modelmaker(numerical_2,y_categorical_selction,i)[0])

In [27]:
selected_categoricals=dfCat.copy()
selected_categoricals=selected_categoricals[catordered.index[:10]]

In [28]:
selected_cat_df=pd.concat([numerical_2,selected_categoricals], axis=1)
selected_cat_df

,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,Neighborhood,Exterior1st,Exterior2nd,Condition2,ExterQual,SaleType,RoofMatl,BsmtQual,KitchenQual,MiscFeature
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,196.0,706,0,150,856,...,CollgCr,VinylSd,VinylSd,Norm,Gd,WD,CompShg,Gd,Gd,NaN
2,20,9600,6,8,1976,0.0,978,0,284,1262,...,Veenker,MetalSd,MetalSd,Norm,TA,WD,CompShg,Gd,TA,NaN
3,60,11250,7,5,2002,162.0,486,0,434,920,...,CollgCr,VinylSd,VinylSd,Norm,Gd,WD,CompShg,Gd,Gd,NaN
4,70,9550,7,5,1970,0.0,216,0,540,756,...,Crawfor,Wd Sdng,Wd Shng,Norm,TA,WD,CompShg,TA,Gd,NaN
5,60,14260,8,5,2000,350.0,655,0,490,1145,...,NoRidge,VinylSd,VinylSd,Norm,Gd,WD,CompShg,Gd,Gd,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,6,5,2000,0.0,0,0,953,953,...,Gilbert,VinylSd,VinylSd,Norm,TA,WD,CompShg,Gd,TA,NaN
1457,20,13175,6,6,1988,119.0,790,163,589,1542,...,NWAmes,Plywood,Plywood,Norm,TA,WD,CompShg,Gd,TA,NaN
1458,70,9042,7,9,2006,0.0,275,0,877,1152,...,Crawfor,CemntBd,CmentBd,Norm,Ex,WD,CompShg,TA,Gd,Shed


In [29]:
selected_cat_df=pd.get_dummies(selected_cat_df)

In [30]:

print(modelmaker(selected_cat_df,y_categorical_selction,2))
print(modelmaker(numerical_2,y_categorical_selction,2))
    

(np.float64(0.21904942512150902), LinearRegression())
(np.float64(0.15849173904374095), LinearRegression())


so we lost? to our selves?

who decided that?

In [31]:
scorecat=[]
for i in range(1,1000):
    scorecat.append(modelmaker(selected_cat_df,y_categorical_selction,i)[0])

In [32]:
pd.DataFrame({'scorecat': scorecat, 'scorenum': scorenum}).mean()

scorecat    0.174447
scorenum    0.276621
dtype: float64

it seems klike we did not

In [33]:
x_num=df_to_pred_num.copy()
x_num=x_num.drop(['1stFlrSF', 'YearBuilt', 'GarageYrBlt', 'LotFrontage', 'HalfBath', 'ScreenPorch', 'EnclosedPorch', 'BsmtFullBath', 'OpenPorchSF', '3SsnPorch', 'MoSold', 'PoolArea', 'YrSold', 'KitchenAbvGr'], axis=1)
x_num=addhascolumns(x_num,zeromeanslack)
x_num=x_num.fillna(x_num.median())

x_cat=df_to_pred_cat.copy()
x_cat=x_cat[catordered.index[:10]]

x_final=pd.concat([x_num,x_cat], axis=1)
x_final=pd.get_dummies(x_final)


unseenikwhy=list(set(selected_cat_df.columns) - set(x_final.columns))

x_final[unseenikwhy]=0

x_final = x_final[selected_cat_df.columns]

OpenPorchSF wasn't found
EnclosedPorch wasn't found
ScreenPorch wasn't found
PoolArea wasn't found
3SsnPorch wasn't found


In [93]:
n=randint(1,100)
modelcat=modelmaker(selected_cat_df,y_categorical_selction,n)

In [94]:
print(n,modelcat)
createsubmissionfie("1_10",modelcat[1],x_final)
#67

84 (np.float64(0.1372175913709313), LinearRegression())
Archivo de submission creado exitosamente!


Now im selecting the categorical variables manually, and cleaning outliers for the training

In [36]:
slectedcats=['Neighborhood','MasVnrType','ExterQual','BsmtQual','HeatingQC','KitchenQual','GarageFinish']

In [37]:
refinedcat=dfCat.copy()
refinedcat=refinedcat[slectedcats]


In [38]:
numerical_refined=numerical_2

numerical_refined['SalePrice']=y_categorical_selction
numerical_refined = numerical_refined[numerical_refined['SalePrice'] < numerical_refined['SalePrice'].quantile(0.95)]

In [39]:
refinedcat = refinedcat.loc[numerical_refined.index]


In [40]:
refinedc_x_train=pd.concat([numerical_refined,refinedcat], axis=1)
refinedc_x_train=pd.get_dummies(refinedc_x_train,drop_first=True)
y_refined=refinedc_x_train['SalePrice']
refinedc_x_train=refinedc_x_train.drop('SalePrice',axis=1)
refinedc_x_train

,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,BsmtQual_TA,HeatingQC_Fa,HeatingQC_Gd,HeatingQC_Po,HeatingQC_TA,KitchenQual_Fa,KitchenQual_Gd,KitchenQual_TA,GarageFinish_RFn,GarageFinish_Unf
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,196.0,706,0,150,856,...,False,False,False,False,False,False,True,False,True,False
2,20,9600,6,8,1976,0.0,978,0,284,1262,...,False,False,False,False,False,False,False,True,True,False
3,60,11250,7,5,2002,162.0,486,0,434,920,...,False,False,False,False,False,False,True,False,True,False
4,70,9550,7,5,1970,0.0,216,0,540,756,...,True,False,True,False,False,False,True,False,False,True
5,60,14260,8,5,2000,350.0,655,0,490,1145,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,6,5,2000,0.0,0,0,953,953,...,False,False,False,False,False,False,False,True,True,False
1457,20,13175,6,6,1988,119.0,790,163,589,1542,...,False,False,False,False,True,False,False,True,False,True
1458,70,9042,7,9,2006,0.0,275,0,877,1152,...,True,False,False,False,False,False,True,False,True,False


In [41]:
refined_pred=df_to_pred_cat.copy()
refined_pred=refined_pred[slectedcats]
refined_pred_num=x_num.copy()

refinedc_x_test=pd.concat([refined_pred_num,refined_pred], axis=1)
refinedc_x_test=pd.get_dummies(refinedc_x_test,drop_first=True)
refinedc_x_test

,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,BsmtQual_TA,HeatingQC_Fa,HeatingQC_Gd,HeatingQC_Po,HeatingQC_TA,KitchenQual_Fa,KitchenQual_Gd,KitchenQual_TA,GarageFinish_RFn,GarageFinish_Unf
Id,,,,,,,,,,,,,,,,,,,,,
1461,20,11622,5,6,1961,0.0,468.0,144.0,270.0,882.0,...,True,False,False,False,True,False,False,True,False,True
1462,20,14267,6,6,1958,108.0,923.0,0.0,406.0,1329.0,...,True,False,False,False,True,False,True,False,False,True
1463,60,13830,5,5,1998,0.0,791.0,0.0,137.0,928.0,...,False,False,True,False,False,False,False,True,False,False
1464,60,9978,6,6,1998,20.0,602.0,0.0,324.0,926.0,...,True,False,False,False,False,False,True,False,False,False
1465,120,5005,8,5,1992,0.0,263.0,0.0,1017.0,1280.0,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2915,160,1936,4,7,1970,0.0,0.0,0.0,546.0,546.0,...,True,False,True,False,False,False,False,True,False,False
2916,160,1894,4,5,1970,0.0,252.0,0.0,294.0,546.0,...,True,False,False,False,True,False,False,True,False,True
2917,20,20000,5,7,1996,0.0,1224.0,0.0,0.0,1224.0,...,True,False,False,False,False,False,False,True,False,True


In [42]:
refined_model=modelmaker(refinedc_x_train,y_refined,67)

In [43]:
createsubmissionfie("1_11",refined_model[1],refinedc_x_test)

Archivo de submission creado exitosamente!


this model got .1535

In [44]:
refinedmodels=[]
for i in range (0,1000):
    refinedmodels.append(modelmaker(refinedc_x_train,y_refined,i)[0])


In [45]:
idx=pd.DataFrame(refinedmodels).idxmin()[0]
idx

np.int64(738)

In [46]:
best=modelmaker(refinedc_x_train,y_refined,idx)
best[0]


np.float64(0.1133780029061371)

In [47]:
createsubmissionfie("1_12",best[1],refinedc_x_test)

Archivo de submission creado exitosamente!


this one got .1585

this is the end of experiments 2, now i will rok on EDA2 and Model 2 wich will use XGBoost

before that i made a new order for my categorical variables, so im training many models based on that order:

In [48]:
numericalnosl=numerical_2.drop('SalePrice',axis=1)
scoresforn=pd.DataFrame()
models=[]
ys=[]
for n in range(0,len(ord_bsd_on_median_ns)):
    categoricalsforthisrun=dfCat.copy()
    categoricalsforthisrun=categoricalsforthisrun[ord_bsd_on_median_ns.index[:n]]
    boththisrun=pd.concat([numericalnosl,categoricalsforthisrun], axis=1)
    boththisrun = boththisrun.loc[numerical_refined.index]
    boththisrun=pd.get_dummies(boththisrun)


    thisruncattest=df_to_pred_cat.copy()
    thisruncattest=thisruncattest[ord_bsd_on_median_ns.index[:n]]

    thisrunxtest=pd.concat([refined_pred_num,thisruncattest], axis=1)
    thisrunxtest=pd.get_dummies(thisrunxtest,drop_first=True)

    unseenintherun=list(set(boththisrun.columns) - set(thisrunxtest.columns))

    thisrunxtest[unseenintherun]=0
    thisrunxtest = thisrunxtest[boththisrun.columns]
    
    nscores=pd.Series()
    bestscore=100
    for run in range(0,3):
        model=modelmaker(boththisrun,y_refined,run)
        if run==2:
            models.append(model)
        nscores[run]=model[0]
    scoresforn=pd.concat([scoresforn,nscores.rename(n)], axis=1)

    ys.append(thisrunxtest)





C:\Users\evera\AppData\Local\Temp\ipykernel_7776\2710784347.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  thisrunxtest[unseenintherun]=0
C:\Users\evera\AppData\Local\Temp\ipykernel_7776\2710784347.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  thisrunxtest[unseenintherun]=0
C:\Users\evera\AppData\Local\Temp\ipykernel_7776\2710784347.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at 

In [49]:
thisrunxtest

,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,Alley_Pave,Fence_GdPrv,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Street_Grvl,Street_Pave,LandSlope_Gtl,LandSlope_Mod,LandSlope_Sev
Id,,,,,,,,,,,,,,,,,,,,,
1461,20,11622,5,6,1961,0.0,468.0,144.0,270.0,882.0,...,False,0,False,True,False,0,True,0,False,False
1462,20,14267,6,6,1958,108.0,923.0,0.0,406.0,1329.0,...,False,0,False,False,False,0,True,0,False,False
1463,60,13830,5,5,1998,0.0,791.0,0.0,137.0,928.0,...,False,0,False,True,False,0,True,0,False,False
1464,60,9978,6,6,1998,20.0,602.0,0.0,324.0,926.0,...,False,0,False,False,False,0,True,0,False,False
1465,120,5005,8,5,1992,0.0,263.0,0.0,1017.0,1280.0,...,False,0,False,False,False,0,True,0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2915,160,1936,4,7,1970,0.0,0.0,0.0,546.0,546.0,...,False,0,False,False,False,0,True,0,False,False
2916,160,1894,4,5,1970,0.0,252.0,0.0,294.0,546.0,...,False,0,False,False,False,0,True,0,False,False
2917,20,20000,5,7,1996,0.0,1224.0,0.0,0.0,1224.0,...,False,0,False,False,False,0,True,0,False,False


In [50]:
boththisrun

,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,Alley_Pave,Fence_GdPrv,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Street_Grvl,Street_Pave,LandSlope_Gtl,LandSlope_Mod,LandSlope_Sev
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,196.0,706,0,150,856,...,False,False,False,False,False,False,True,True,False,False
2,20,9600,6,8,1976,0.0,978,0,284,1262,...,False,False,False,False,False,False,True,True,False,False
3,60,11250,7,5,2002,162.0,486,0,434,920,...,False,False,False,False,False,False,True,True,False,False
4,70,9550,7,5,1970,0.0,216,0,540,756,...,False,False,False,False,False,False,True,True,False,False
5,60,14260,8,5,2000,350.0,655,0,490,1145,...,False,False,False,False,False,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,6,5,2000,0.0,0,0,953,953,...,False,False,False,False,False,False,True,True,False,False
1457,20,13175,6,6,1988,119.0,790,163,589,1542,...,False,False,False,True,False,False,True,True,False,False
1458,70,9042,7,9,2006,0.0,275,0,877,1152,...,False,True,False,False,False,False,True,True,False,False


In [51]:
scoresforn

,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0.163318,0.151821,0.158074,0.158186,0.159139,0.159343,0.164847,0.162284,0.162318,0.165661,...,0.143488,0.143273,0.140284,0.139584,0.139835,0.139149,0.140540,0.139786,0.135077,0.131323
1,0.134116,0.128766,0.130001,0.147286,0.146702,0.141912,0.139340,0.143734,0.145640,0.145579,...,0.137212,0.140080,0.139709,0.136607,0.131999,0.134120,0.135859,0.134098,0.131374,0.127504
2,0.207857,0.162649,0.157829,0.159858,0.159109,0.157617,0.160835,0.159820,0.160350,0.160954,...,0.138615,0.137777,0.137903,0.136154,0.139331,0.138679,0.138554,0.139354,0.137842,0.153120


In [52]:
moves=pd.DataFrame()
last=scoresforn.columns[0]
for columna in scoresforn.columns:
    moves[columna]=scoresforn[last]-scoresforn[columna]
    last=columna

In [53]:
moves

,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0.0,0.011497,-0.006254,-0.000111,-0.000953,-0.000203,-0.005505,0.002564,-0.000034,-0.003344,...,-0.001300,0.000216,0.002989,0.000699,-0.000251,0.000687,-0.001392,0.000754,0.004710,0.003753
1,0.0,0.005350,-0.001235,-0.017285,0.000584,0.004791,0.002572,-0.004395,-0.001906,0.000061,...,-0.000616,-0.002868,0.000370,0.003103,0.004608,-0.002121,-0.001739,0.001761,0.002725,0.003869
2,0.0,0.045207,0.004821,-0.002029,0.000749,0.001492,-0.003218,0.001015,-0.000530,-0.000604,...,-0.000327,0.000838,-0.000126,0.001749,-0.003177,0.000652,0.000125,-0.000800,0.001513,-0.015278


In [95]:
positive_effects = moves.loc[:, moves.mean() < 0]
positive_effects

,2,3,6,7,8,9,11,12,16,17,...,22,23,24,26,31,33,34,38,39,42
0,-0.006254,-0.000111,-0.005505,0.002564,-0.000034,-0.003344,0.001145,0.000537,0.001014,0.000852,...,-0.002928,-0.005537,-0.007351,-0.003940,0.000045,-0.001300,0.000216,0.000687,-0.001392,0.003753
1,-0.001235,-0.017285,0.002572,-0.004395,-0.001906,0.000061,-0.003343,-0.000315,0.001146,-0.006687,...,-0.002076,-0.000053,-0.002996,-0.001311,-0.000325,-0.000616,-0.002868,-0.002121,-0.001739,0.003869
2,0.004821,-0.002029,-0.003218,0.001015,-0.000530,-0.000604,-0.000334,-0.002141,-0.002249,-0.001013,...,0.001479,-0.000686,-0.000621,-0.001001,-0.000375,-0.000327,0.000838,0.000652,0.000125,-0.015278


In [96]:
positive_effects_list=list(positive_effects.columns)
positive_effects_list

[2,
 3,
 6,
 7,
 8,
 9,
 11,
 12,
 16,
 17,
 18,
 19,
 21,
 22,
 23,
 24,
 26,
 31,
 33,
 34,
 38,
 39,
 42]

In [97]:
finalcat=dfCat.copy()
finalcat=finalcat.drop(finalcat.columns[positive_effects_list],axis=1)

In [98]:
finalcat

,MSZoning,Street,LandContour,Utilities,Condition2,RoofStyle,RoofMatl,Exterior1st,Foundation,BsmtFinType2,HeatingQC,CentralAir,Electrical,KitchenQual,FireplaceQu,GarageQual,GarageCond,PavedDrive,MiscFeature,SaleType
Id,,,,,,,,,,,,,,,,,,,,
1,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,VinylSd,PConc,Unf,Ex,Y,SBrkr,Gd,NaN,TA,TA,Y,NaN,WD
2,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,MetalSd,CBlock,Unf,Ex,Y,SBrkr,TA,TA,TA,TA,Y,NaN,WD
3,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,VinylSd,PConc,Unf,Ex,Y,SBrkr,Gd,TA,TA,TA,Y,NaN,WD
4,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,Wd Sdng,BrkTil,Unf,Gd,Y,SBrkr,Gd,Gd,TA,TA,Y,NaN,WD
5,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,VinylSd,PConc,Unf,Ex,Y,SBrkr,Gd,TA,TA,TA,Y,NaN,WD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,VinylSd,PConc,Unf,Ex,Y,SBrkr,TA,TA,TA,TA,Y,NaN,WD
1457,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,Plywood,CBlock,Rec,TA,Y,SBrkr,TA,TA,TA,TA,Y,NaN,WD
1458,RL,Pave,Lvl,AllPub,Norm,Gable,CompShg,CemntBd,Stone,Unf,Ex,Y,SBrkr,Gd,Gd,TA,TA,Y,Shed,WD


In [99]:
finalboth=pd.concat([numericalnosl,finalcat], axis=1)
finalboth = finalboth.loc[numerical_refined.index]
finalboth=pd.get_dummies(finalboth)


finalcattest=df_to_pred_cat.copy()
finalcattest=finalcattest.drop(finalcattest.columns[positive_effects.columns],axis=1)

finalxtest=pd.concat([refined_pred_num,finalcattest], axis=1)
finalxtest=pd.get_dummies(finalxtest,drop_first=True)

unseenfinal=list(set(finalboth.columns) - set(finalxtest.columns))

finalxtest[unseenfinal]=0
finalxtest = finalxtest[finalboth.columns]

In [100]:
finalboth

,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,MiscFeature_TenC,SaleType_COD,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,196.0,706,0,150,856,...,False,False,False,False,False,False,False,False,False,True
2,20,9600,6,8,1976,0.0,978,0,284,1262,...,False,False,False,False,False,False,False,False,False,True
3,60,11250,7,5,2002,162.0,486,0,434,920,...,False,False,False,False,False,False,False,False,False,True
4,70,9550,7,5,1970,0.0,216,0,540,756,...,False,False,False,False,False,False,False,False,False,True
5,60,14260,8,5,2000,350.0,655,0,490,1145,...,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,6,5,2000,0.0,0,0,953,953,...,False,False,False,False,False,False,False,False,False,True
1457,20,13175,6,6,1988,119.0,790,163,589,1542,...,False,False,False,False,False,False,False,False,False,True
1458,70,9042,7,9,2006,0.0,275,0,877,1152,...,False,False,False,False,False,False,False,False,False,True


In [101]:
finalxtest

,MSSubClass,LotArea,OverallQual,OverallCond,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,MiscFeature_TenC,SaleType_COD,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD
Id,,,,,,,,,,,,,,,,,,,,,
1461,20,11622,5,6,1961,0.0,468.0,144.0,270.0,882.0,...,0,0,False,False,False,False,False,False,False,True
1462,20,14267,6,6,1958,108.0,923.0,0.0,406.0,1329.0,...,0,0,False,False,False,False,False,False,False,True
1463,60,13830,5,5,1998,0.0,791.0,0.0,137.0,928.0,...,0,0,False,False,False,False,False,False,False,True
1464,60,9978,6,6,1998,20.0,602.0,0.0,324.0,926.0,...,0,0,False,False,False,False,False,False,False,True
1465,120,5005,8,5,1992,0.0,263.0,0.0,1017.0,1280.0,...,0,0,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2915,160,1936,4,7,1970,0.0,0.0,0.0,546.0,546.0,...,0,0,False,False,False,False,False,False,False,True
2916,160,1894,4,5,1970,0.0,252.0,0.0,294.0,546.0,...,0,0,False,False,False,False,False,False,False,True
2917,20,20000,5,7,1996,0.0,1224.0,0.0,0.0,1224.0,...,0,0,False,False,False,False,False,False,False,True


In [110]:
finalmodel=modelmaker(finalboth,y_refined,9)

In [109]:
finalmodel

(np.float64(0.1450908478458635), LinearRegression())

In [104]:
createsubmissionfie("1_14",finalmodel[1],finalxtest)

Archivo de submission creado exitosamente!


In [105]:
createsubmissionfie("1_15",models[6][1],ys[6])

Archivo de submission creado exitosamente!
